# 02 · Reciprocal space — the resolution kernel

The kernel Res(**q**) is the instrument's acceptance: the probability that a ray scattered with reciprocal-space deviation **q** = (q₁, q₂, q₃) clears condenser, objective and bandwidth to reach the detector. `dfxm-bootstrap` estimates it by Monte-Carlo ray sampling ([Poulsen 2021](../docs/references.md#poulsen-2021), [Poulsen 2017](../docs/references.md#poulsen-2017)); the forward model then weights every voxel's local q against this table ([Borgi 2024](../docs/references.md#borgi-2024) §3).

In [ ]:
%matplotlib inline
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

import dfxm_geo.direct_space.forward_model as fm

HERE = Path.cwd()
assert HERE.name == "examples", "Run this notebook from the examples/ folder"
IMG, OUT, KERNEL_DIR = HERE / "img", HERE / "out_02", HERE / "kernel"
for d in (IMG, OUT, KERNEL_DIR):
    d.mkdir(exist_ok=True)
fm.pkl_fpath = KERNEL_DIR.resolve().as_posix() + "/"

KERNEL_FILE = KERNEL_DIR / "Resq_i_h-1_k1_l-1_17keV_tutorial.npz"
if not KERNEL_FILE.exists():
    boot = OUT / "bootstrap.toml"
    boot.write_text(
        "[reciprocal]\nNrays = 2_000_000\nseed = 0\nbeamstop = false\n", encoding="utf-8"
    )
    subprocess.run(
        [
            sys.executable,
            "-c",
            "from dfxm_geo.reciprocal_space.kernel import cli_main; cli_main()",
            "--config",
            str(boot),
            "--output",
            str(KERNEL_FILE),
        ],
        check=True,
    )
print("kernel:", KERNEL_FILE.name)

## Inside the npz

The kernel ships its full provenance — reflection, energy, Bragg angle, ray count, grid geometry and instrument parameters — alongside the 3-D lookup table itself.

In [ ]:
k = np.load(KERNEL_FILE)
for key in (
    "hkl",
    "keV",
    "theta",
    "Nrays",
    "npoints1",
    "npoints2",
    "npoints3",
    "qi1_range",
    "qi2_range",
    "qi3_range",
    "beamstop",
    "seed",
):
    print(f"{key:12s} = {k[key]}")
R = k["Resq_i"]
print("LUT shape:", R.shape, "| nonzero voxels:", int((R > 0).sum()))

## The acceptance as a point cloud

Scattering the occupied voxels shows the elongated acceptance volume the instrument carves out of reciprocal space (compare `docs/img/reciprocal_pointcloud_50k.png`, rendered at 10⁸ rays).

In [ ]:
q1 = np.linspace(-k["qi1_range"], k["qi1_range"], R.shape[0])
q2 = np.linspace(-k["qi2_range"], k["qi2_range"], R.shape[1])
q3 = np.linspace(-k["qi3_range"], k["qi3_range"], R.shape[2])
idx = np.argwhere(0.01 * R.max() < R)
rng = np.random.default_rng(0)
idx = idx[rng.choice(len(idx), size=min(50_000, len(idx)), replace=False)]
w = R[idx[:, 0], idx[:, 1], idx[:, 2]]

fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(projection="3d")
ax.scatter(q1[idx[:, 0]], q2[idx[:, 1]], q3[idx[:, 2]], c=w, s=1, cmap="viridis", alpha=0.3)
ax.set_xlabel("$q_1$")
ax.set_ylabel("$q_2$")
ax.set_zlabel("$q_3$")
fig.savefig(IMG / "02_reciprocal_space_preview.png", dpi=110, bbox_inches="tight")

## MC kernel vs the closed-form (analytic) backend

For the no-beamstop regime there is also a closed-form backend (v2.1.0) — same physics, zero sampling noise, no kernel file; the ±140 µrad vertical-divergence clip both backends apply is the condenser's physical aperture ([Carlsen 2022](../docs/references.md#carlsen-2022)). At only 2 M rays the MC centre-of-mass maps are visibly noisier — that is the trade, and `[reciprocal] backend` is the switch. Full parity study: [docs/resolution-backend-comparison.md](../docs/resolution-backend-comparison.md).

In [ ]:
from dfxm_geo.analysis.mosaicity import compute_com_maps
from dfxm_geo.io.hdf5 import load_h5_scan
from dfxm_geo.pipeline import SimulationConfig, run_simulation

BASE = """
[scan.phi]
range = 6e-4
steps = 9

[scan.chi]
range = 2e-3
steps = 9

[io]
include_perfect_crystal = false
write_strain_provenance = false

[postprocess]
enabled = false
"""
com = {}
for backend in ("mc", "analytic"):
    cfg_file = OUT / f"mosa_{backend}.toml"
    cfg_file.write_text(
        f'[reciprocal]\nbackend = "{backend}"\nbeamstop = false\n' + BASE, encoding="utf-8"
    )
    res = run_simulation(SimulationConfig.from_toml(cfg_file), OUT / backend)
    _, stack, h, w_ = load_h5_scan(res["h5_path"], phi_steps=9, chi_steps=9)
    com[backend] = compute_com_maps(stack, phi_range=6e-4, phi_steps=9, chi_range=2e-3, chi_steps=9)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
vmax = 1e-4
im0 = axes[0].imshow(com["mc"][0], vmin=-vmax, vmax=vmax, cmap="RdBu_r")
axes[0].set_title("φ-COM, MC kernel (2 M rays)")
axes[1].imshow(com["analytic"][0], vmin=-vmax, vmax=vmax, cmap="RdBu_r")
axes[1].set_title("φ-COM, analytic backend")
axes[2].imshow(com["mc"][0] - com["analytic"][0], vmin=-vmax, vmax=vmax, cmap="RdBu_r")
axes[2].set_title("difference (MC sampling noise)")
for ax in axes:
    ax.axis("off")
fig.colorbar(im0, ax=axes, shrink=0.8, label="rad")

Production kernels (10⁸ rays) drive that difference toward zero — see [docs/resolution-backend-comparison.md](../docs/resolution-backend-comparison.md). Next: [03 · Dislocations & contrast](03_dislocations_and_contrast.ipynb).